# PixelTable and BinTable

In [1]:
# Import packages
import os
import tempfile

import polars as pl
import hicona as hi

# Data paths 
COOLER_PATH = "../data/4DNFIIG4IWKW_10k.cool"

The `PixelTable` is the main class around which HiCONA revolves. If you are not planning on extending HiCONA's functionalities or trying niche corner-cases, once you have created a `PixelTable` object you will likely work exclusively on it. Each `PixelTable` is associated with a `BinTable`, containing information on the binning.

## The Table storage model
Iterating an entire cooler file might be very inefficient if you are interested in a subset of the pixels, and especially so if those pixels are not contiguous in the file. Add the fact that you might want to manipulate the pixels (normalize, for instance) without keeping them all into memory, and you see why defaulting to reading from cooler might not be always the most convenient choice. The way HiCONA addresses this issue is through `Table` objects.

The `Table` class allows to handle read/write operations to a temporary local storage. When a `Table` instance is created, a partitioned [parquet storage](https://parquet.apache.org/) is created inside the `tmp` folder of your system; when the instance is deleted (or garbage collected) the storage is also deleted. This allows you to store data in a fast temporary local storage whose content can be modified without affecting the original copy of the data. 


## Creating a PixelTable

The `PixelTable` and the `BinTable` classes are specialized subclasses of `Table` used to handle, respectively, pixels and bins. A `PixelTable` contains pixels in the `bin1_id`, `bin2_id`, `count` format (plus pixel level annotations, see the **PixelTable annotation** section). The `BinTable` contains a full genomic binning, meanining that it has the columns `bin_id`, `chrom`, `start`, `end` plus any number of bin level annotation columns.

The main way you will create a `PixelTable` is from a `HiconaCooler` by specifying a genomic region.

In [2]:
handle = hi.HiconaCooler(COOLER_PATH)
pix_table = handle.get_pixel_table("chr10")

This step might be a bit slow for very big files, since a full pass of the cooler is required and the data needs to be written to the temporary storage.

A `BinTable` can be created in a similar way. No region needs to be specified since the full bin table from the cooler is copied (the full genome binning is required for some features such as `PixelTable` comparison, subsetting and plotting).

In [3]:
bin_table = handle.get_bin_table()

Notice that, when creating a `PixelTable`, you do not need to create a `BinTable` directly. This is because the `BinTable` containing information on the binning used for the `PixelTable` is automatically generated and stored in the `bins` attribute of the `PixelTable`.

In [4]:
type(pix_table.bins)

hicona._core.table.BinTable

Both `BinTable` and `PixelTable` instances can be created directly from iterables of bins and pixels (similar to `cooler.create_cooler`).

<div class=note>
    From here on, we will be talking mostly about PixelTable, since BinTable is mostly used internally by PixelTable, rather than accessed
    directly. You can assume that, for all non-pixel specific attributes and methods of PixelTable, there are analogous ones for BinTable.
    For more detail, please refer to the API documentation.
</div>

## Extracting data from a PixelTable

The `PixelTable` class provides three methods to access its data: `get_chunks`, `get_dataframe`, `get_matrix`.

`get_chunks`, as the name suggests, returns an generator of chunks of the `PixelTable`. The full table can be returned or a subset of it by providing a genomic region. Furthermore, different parameters can be returned to customize these chunks:

- `annotate`: whether to return `bin1_id`, `bin2_id`, `count` or the expended format (see `cooler.annotate`).
- `balance`: apply ICE normalization. Requires a `weight` column to be present in the `BinTable`.
- `chunk_size`: dimension of the chunks to fetch.
- `dtype`: the type of dataframe returned (`pandas` or `polars`)
- `strategies`: any user defined function which takes a generator of chunks as input and returns a new one as output.

In [5]:
spacer = "-" * 80 + "\n"

print("Default chunks")
for chunk in pix_table.get_chunks():
    print(chunk.head())
    print(spacer)
    break

print("Region subset and annotated chunks")
for chunk in pix_table.get_chunks("chr10:0-1000000", annotate=True):
    print(chunk.head())
    print(spacer)
    break

print("Pandas chunks of size 500")
for chunk in pix_table.get_chunks(dtype="pandas", chunk_size=500):
    print(chunk.head())
    print(spacer)
    break

def mock_strategy(chunks):
    for chunk in chunks:
        yield chunk.with_columns(pl.col("count") + 1000)

print("Custom strategy chunks")
for chunk in pix_table.get_chunks(strategies=mock_strategy):
    print(chunk.head())
    print(spacer)
    break

Default chunks
shape: (5, 3)
┌─────────┬─────────┬───────┐
│ bin1_id ┆ bin2_id ┆ count │
│ ---     ┆ ---     ┆ ---   │
│ i64     ┆ i64     ┆ i32   │
╞═════════╪═════════╪═══════╡
│ 167493  ┆ 167493  ┆ 3     │
│ 167493  ┆ 167496  ┆ 3     │
│ 167493  ┆ 167497  ┆ 3     │
│ 167493  ┆ 167498  ┆ 3     │
│ 167493  ┆ 167499  ┆ 6     │
└─────────┴─────────┴───────┘
--------------------------------------------------------------------------------

Region subset and annotated chunks
shape: (5, 9)
┌─────────┬─────────┬───────┬────────┬───┬───────┬────────┬────────┬───────┐
│ bin1_id ┆ bin2_id ┆ count ┆ chrom1 ┆ … ┆ end1  ┆ chrom2 ┆ start2 ┆ end2  │
│ ---     ┆ ---     ┆ ---   ┆ ---    ┆   ┆ ---   ┆ ---    ┆ ---    ┆ ---   │
│ i64     ┆ i64     ┆ i32   ┆ str    ┆   ┆ i32   ┆ str    ┆ i32    ┆ i32   │
╞═════════╪═════════╪═══════╪════════╪═══╪═══════╪════════╪════════╪═══════╡
│ 167493  ┆ 167493  ┆ 3     ┆ chr10  ┆ … ┆ 20000 ┆ chr10  ┆ 10000  ┆ 20000 │
│ 167493  ┆ 167496  ┆ 3     ┆ chr10  ┆ … ┆ 20000

<div class=warn>
Internally, HiCONA uses <a href="https://pola.rs/">polars</a> to handle datasets for better performance. Since polars does not support index columns, the `bin_id` column in the `BinTable` is an explicit column, rather than the index itself. When fetching dataframes in pandas, HiCONA provides no guarantee that the index matches the `bin_id` column.
</div>

`get_dataframe` is simply `get_chunks` with concatenation on top. `region`, `annotate` and `dtype` parameters are natively supported by the method. All other parameters to pass to the internal call to `get_chunks` should be provided as a dictionary in the `selection_kwargs` parameter.

In [6]:
pix_table.get_dataframe("chr10:100000-200000")

bin1_id,bin2_id,count
i64,i64,i32
167502,167502,594
167502,167503,243
167502,167504,167
167502,167505,66
167502,167506,50
…,…,…
167509,167510,196
167509,167511,142
167510,167510,495


`get_matrix` is instead used to return a 2D-numpy array. Again, this supports the `region` parameter as well as `selection_kwargs`, but it also provides method-specific parameters, namely:

- `value_col`: the values to put in the cells of the matrix. By default `counts` but could be clustering labels and others.
- `mode`: whether to return a full, upper triangular or lower triangular matrix.
- `mask_diagonal`: whether to fill the main diagonal with nans.
        

In [7]:
matrix = pix_table.get_matrix("chr10:0-1000000", mode="upper")
print(matrix)

[[      nan       nan       nan ...       nan       nan       nan]
 [      nan 3.000e+00       nan ...       nan       nan 1.000e+00]
 [      nan       nan 1.000e+00 ... 1.000e+00       nan       nan]
 ...
 [      nan       nan       nan ... 9.290e+02 4.930e+02 1.280e+02]
 [      nan       nan       nan ...       nan 1.290e+03 4.890e+02]
 [      nan       nan       nan ...       nan       nan 1.088e+03]]


<div class=warn>
Be careful since matrices for big genomic regions can be quite heavy in memory.
</div>

## Annotating a PixelTable

`PixelTable` provides methods to add annotations to the bins and the pixels, `add_bin_annotation` and `add_pix_annotation` respectively. These methods are particularly useful when you do not want to pollute your cooler file with annotations you will use just a couple of times. `add_bin_annotation` works in the same way as `HiconaCooler.bin_annotation_add`, see the `HiconaCooler` notebook for examples. `add_pix_annotation` instead works by simply performing a join between the pixels and the provided annotation on the columns `bin1_id` and `bin2_id`.

## Creating a persistent PixelTable

We introduced the `PixelTable` class saying that they create temporary copies of the data. Sometimes though, you might want to save your current table, maybe because performing clustering on it took a long time and you do not want to repeat it every time. To do so, `PixelTable` provides two methods, `save` and `load`. 

<div class=note>
    `save` creates a permanent snapshot of the `PixelTable` at some provided path; further changes of the `PixelTable` do not affect the 
    saved snapshot. `load` creates a new `PixelTable` instance whose data is backed by a copy of the snapshot in the temporary folder.
    This means that changes on the loaded object do not affect the snapshot and must be saved to a new snapshot if needed.
</div>

In [8]:
with tempfile.TemporaryDirectory() as tmp_dir_name:
    snapshot_path = os.path.join(tmp_dir_name, "test_table")

    pix_table.save(snapshot_path)
    reloaded = pix_table.load(snapshot_path)


In [9]:
from IPython.core.display import HTML

def css_styling():
    styles = open("../styles/custom.css", "r").read()
    return HTML(styles)
css_styling()
